<!-- paper-paired-role-banner -->
> 🟩 **정답 코드 · 오른쪽 창** — 왼쪽과 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 구현 이유와 텐서 shape를 설명해 보세요.

# 07. LSTM Sequence to Sequence — Sequence to Sequence Learning with Neural Networks

- **원 논문:** [Sequence to Sequence Learning with Neural Networks](https://arxiv.org/abs/1409.3215)
- **저자 / 발표:** Ilya Sutskever, Oriol Vinyals, Quoc V. Le · NeurIPS 2014 (2014)
- **난이도 / 예상 시간:** 중급 · 약 75분
- **선수 지식:** LSTM, teacher forcing, cross entropy, autoregressive decoding

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

LSTM encoder의 마지막 hidden/cell state를 LSTM decoder의 초기 상태로 넘기고, BOS 입력과 한 칸 이동한 target으로 조건부 다음-token 확률을 학습한다. 논문의 source 순서 뒤집기 기법과 greedy autoregressive decoding까지 작은 복사 과제로 확인한다.

**원 논문과 다른 것**

원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해 | `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL | decoder input과 target이 정확히 한 칸 어긋나는지 assert |
| §2: 첫 LSTM의 마지막 state가 고정 길이 표현 v를 구성 | `Seq2Seq.forward`가 encoder `(h, c)`를 decoder 초기 상태로 전달 | logit shape와 encoder/decoder state shape 검사 |
| §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성 | `greedy_decode`가 BOS부터 예측 token을 다시 입력 | token accuracy와 exact-sequence accuracy를 모두 출력 |
| §3.2: source sentence의 단어 순서를 뒤집으면 minimal time lag 감소 | `build_seq2seq_batch`의 `encoder_ids = flip(source_ids)` | 첫 행이 정확히 역순인지 assert |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
p(y_{1:T}\mid x_{1:S})=\prod_{t=1}^{T}
p(y_t\mid y_{<t},c),\qquad c=\operatorname{Encoder}(x_{1:S})
$$

- $c$는 source sequence를 요약한 encoder state, $y_{<t}$는 이전 target token입니다.
- decoder는 teacher forcing으로 학습하고 autoregressive loop로 추론합니다.
- vocabulary, encoder, decoder, sequence loss, greedy decode를 명시적으로 분리합니다.
- token $[B,S]$는 recurrent state $[L,B,H]$를 거쳐 target logit $[B,T,V]$가 됩니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해
- **노트북 구현:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

## 핵심 아이디어

논문의 기본 모형은 한 LSTM이 source를 읽고 마지막 상태로 표현 $v$를 만든 뒤,
다른 LSTM이
$p(T'\mid S)=\prod_t p(y_t\mid v,y_1,\ldots,y_{t-1})$를 모델링한다.
학습 중에는 실제 이전 token을 넣지만(teacher forcing), 평가 때는 자신의 예측을
다시 입력해야 한다. 논문 §3.2처럼 encoder 입력만 뒤집고 정답 순서는 유지한다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

SEED = 707
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
print(ACCELERATOR.summary())
PAD, BOS, EOS = 0, 1, 2
VOCAB_SIZE = 8

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해 / §3.2: source sentence의 단어 순서를 뒤집으면 minimal time lag 감소
- **이 셀의 책임:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL / `build_seq2seq_batch`의 `encoder_ids = flip(source_ids)`
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert / 첫 행이 정확히 역순인지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.cat,ACCELERATOR.move,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
source_ids = torch.tensor(
    [
        [3, 4, 5],
        [4, 5, 6],
        [5, 6, 7],
        [7, 6, 5],
        [3, 6, 4],
        [4, 7, 3],
        [5, 3, 7],
        [6, 4, 3],
    ]
)


def build_seq2seq_batch(source):
    if source.ndim != 2:
        raise ValueError("source must have shape [batch, length]")
    encoder_input = torch.flip(source, dims=(1,))
    batch = source.shape[0]
    bos_column = torch.full((batch, 1), BOS, dtype=torch.long)
    eos_column = torch.full((batch, 1), EOS, dtype=torch.long)
    decoder_target = torch.cat(
        (bos_column, source, eos_column),
        dim=1,
    )
    return encoder_input, decoder_target


encoder_ids, target_ids = build_seq2seq_batch(source_ids)
source_ids, encoder_ids, target_ids = ACCELERATOR.move(
    source_ids, encoder_ids, target_ids
)

assert torch.equal(encoder_ids[0], encoder_ids.new_tensor([5, 4, 3]))
assert torch.all(target_ids[:, 0] == BOS) and torch.all(target_ids[:, -1] == EOS)
print("encoder:", encoder_ids[0].tolist(), "target:", target_ids[0].tolist())

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §2: 첫 LSTM의 마지막 state가 고정 길이 표현 v를 구성
- **이 셀의 책임:** `Seq2Seq.forward`가 encoder `(h, c)`를 decoder 초기 상태로 전달
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** logit shape와 encoder/decoder state shape 검사. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Embedding,nn.LSTM,nn.Linear -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>nn.LSTM(input_size, hidden_size, ...)</code></summary>

#### `nn.LSTM(input_size, hidden_size, ...)`

- **역할:** input·forget·output gate와 cell state를 가진 recurrent 층을 만듭니다.
- **입력·반환:** 입력·hidden 크기와 설정을 받고 sequence module을 반환합니다.
- **이 셀에서 쓰는 이유:** 장기 정보를 cell state로 보존하는 sequence encoder·decoder를 구현합니다.
- **주의점:** hidden state와 cell state 두 개를 전달하며 `batch_first`가 둘의 shape은 바꾸지 않습니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}i_t&=\sigma(W_{ii}x_t+b_{ii}+W_{hi}h_{t-1}+b_{hi}),\\f_t&=\sigma(W_{if}x_t+b_{if}+W_{hf}h_{t-1}+b_{hf}),\\g_t&=\tanh(W_{ig}x_t+b_{ig}+W_{hg}h_{t-1}+b_{hg}),\\o_t&=\sigma(W_{io}x_t+b_{io}+W_{ho}h_{t-1}+b_{ho}),\\c_t&=f_t\odot c_{t-1}+i_t\odot g_t,\qquad h_t=o_t\odot\tanh(c_t)\end{aligned}
$$

- **기호:** $i,f,o$는 input·forget·output gate, $c_t$는 cell state입니다.
- **수식 → 코드:** LSTM module이 각 time step에서 네 gate와 cell·hidden update를 계산합니다.
- **직관:** cell state가 정보를 더하기 경로로 전달해 기본 RNN보다 긴 의존성을 보존합니다.
- **shape 확인:** 입력·출력은 RNN과 같고 마지막 상태는 `(h_n, c_n)` 두 `[L,B,H]` 텐서입니다.
- **공식 문서:** [nn.LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, embedding_dim=16, hidden_dim=32):
        super().__init__()
        self.source_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.target_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.encoder = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.decoder = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def encode(self, source):
        _, state = self.encoder(self.source_embedding(source))
        return state

    def decode_teacher_forced(self, decoder_inputs, state):
        decoded, next_state = self.decoder(self.target_embedding(decoder_inputs), state)
        return self.output(decoded), next_state

    def forward(self, source, decoder_inputs):
        state = self.encode(source)
        logits, _ = self.decode_teacher_forced(decoder_inputs, state)
        return logits


torch.manual_seed(SEED)
model = Seq2Seq(VOCAB_SIZE).to(DEVICE)
probe_logits = model(encoder_ids, target_ids[:, :-1])
assert probe_logits.shape == (len(source_ids), target_ids.shape[1] - 1, VOCAB_SIZE)

### 구현 단계 4 · 목적함수·학습 update

- **원문 위치:** §2, 식 (1): 조건부 확률을 token별 확률의 곱으로 분해
- **이 셀의 책임:** `decoder_inputs`/`next_token_targets` shift와 teacher-forced NLL
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** decoder input과 target이 정확히 한 칸 어긋나는지 assert. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.Adam,Module.parameters,Module.train,Optimizer.zero_grad,F.cross_entropy,Tensor.reshape,Tensor.backward,clip_grad_norm_,Optimizer.step,Tensor.detach,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 11개 · 수식 6개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>model.train(mode=True)</code></summary>

#### `model.train(mode=True)`

- **역할:** 모듈과 하위 모듈을 학습 모드로 전환합니다.
- **입력·반환:** 불리언 모드를 받고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout과 BatchNorm이 학습 시 동작을 사용하도록 합니다.
- **주의점:** gradient를 켜는 함수가 아니므로 `no_grad`와 독립적입니다.
- **공식 문서:** [Module.train](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)</code></summary>

#### `clip_grad_norm_(parameters, max_norm, norm_type=2.0, ...)`

- **역할:** 전체 gradient norm이 한계를 넘으면 같은 비율로 축소합니다.
- **입력·반환:** 파라미터와 최대 norm을 받고 clipping 전 전체 norm을 반환합니다.
- **이 셀에서 쓰는 이유:** RNN·Transformer·RL 학습의 폭주하는 update를 제한합니다.
- **주의점:** `backward()` 뒤 `step()` 전에 호출해야 하며 근본 원인을 없애지는 않습니다.

##### 관련 수식과 코드 연결

$$
g' = g\cdot\min\!\left(1,\frac{c}{\lVert g\rVert_p+\epsilon}\right)
$$

- **기호:** $g$는 모든 parameter gradient를 합친 벡터, $c$는 `max_norm`입니다.
- **수식 → 코드:** 전체 norm이 한계를 넘을 때만 모든 gradient를 같은 비율로 줄입니다.
- **직관:** 방향은 유지하면서 update 크기만 제한합니다.
- **shape 확인:** 각 parameter의 gradient shape은 유지됩니다.
- **공식 문서:** [clip_grad_norm_](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.clip_grad_norm_.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
decoder_inputs = target_ids[:, :-1]
next_token_targets = target_ids[:, 1:]
assert torch.equal(decoder_inputs[:, 1:], next_token_targets[:, :-1])

optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_history = []


def train_seq2seq_step(model, source, inputs, targets, optimizer):
    model.train()
    optimizer.zero_grad()
    logits = model(source, inputs)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        targets.reshape(-1),
    )
    loss.backward()
    gradient_norm = nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=5.0,
    )
    optimizer.step()
    return float(loss.detach()), float(gradient_norm)


gradient_norms = []
for _ in range(180):
    step_loss, gradient_norm = train_seq2seq_step(
        model,
        encoder_ids,
        decoder_inputs,
        next_token_targets,
        optimizer,
    )
    loss_history.append(step_loss)
    gradient_norms.append(gradient_norm)

assert loss_history[-1] < 0.15 * loss_history[0]
assert torch.isfinite(torch.tensor(gradient_norms)).all()
print(f"teacher-forced loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성
- **이 셀의 책임:** `greedy_decode`가 BOS부터 예측 token을 다시 입력
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** token accuracy와 exact-sequence accuracy를 모두 출력. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.inference_mode,Module.eval -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

<details>
<summary><code>model.eval()</code></summary>

#### `model.eval()`

- **역할:** 모듈과 하위 모듈을 평가 모드로 전환합니다.
- **입력·반환:** 인자 없이 호출하고 모듈 자신을 반환합니다.
- **이 셀에서 쓰는 이유:** Dropout을 끄고 BatchNorm의 저장 통계를 사용하게 합니다.
- **주의점:** gradient 기록은 끄지 않으므로 필요하면 `no_grad`도 사용합니다.
- **공식 문서:** [Module.eval](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
@torch.inference_mode()
def greedy_decode(model, source, steps):
    model.eval()
    state = model.encode(source)
    current = torch.full(
        (len(source), 1),
        BOS,
        dtype=torch.long,
        device=source.device,
    )
    predictions = []
    for _ in range(steps):
        logits, state = model.decode_teacher_forced(current, state)
        current = logits[:, -1:].argmax(dim=-1)
        predictions.append(current)
    return torch.cat(predictions, dim=1)


predicted_ids = greedy_decode(model, encoder_ids, target_ids.shape[1] - 1)
expected_ids = target_ids[:, 1:]
token_accuracy = (predicted_ids == expected_ids).float().mean().item()
exact_accuracy = (predicted_ids == expected_ids).all(dim=1).float().mean().item()
assert token_accuracy >= 0.95 and exact_accuracy >= 0.75
print(
    f"token_accuracy={token_accuracy:.1%}, exact_sequence_accuracy={exact_accuracy:.1%}"
)
print(
    "prediction:",
    predicted_ids[0].tolist(),
    "expected:",
    expected_ids[0].tolist(),
)

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §2: 두 번째 LSTM이 v에 조건부로 출력 문장을 생성
- **이 셀의 책임:** `greedy_decode`가 BOS부터 예측 token을 다시 입력
- **Tensor shape 계약:** source [B,S] → recurrent state [L,B,H] → target logits [B,T,V]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** token accuracy와 exact-sequence accuracy를 모두 출력. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
per_sequence = (predicted_ids == expected_ids).all(dim=1).float()
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(loss_history)
axes[0].set(title="teacher-forced NLL", xlabel="step", ylabel="loss")
axes[1].bar(range(len(per_sequence)), per_sequence.tolist())
axes[1].set(title="autoregressive exact match", xlabel="sequence", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

## 해석 체크

teacher-forced loss가 작아도 autoregressive exact match가 낮을 수 있는 이유와,
source를 뒤집는 것이 장거리 의존성의 *시간 간격*을 어떻게 줄이는지 적어 보세요.

## 마무리 해설과 다음 실험

1. 핵심 연산과 구현 위치를 원 논문의 용어로 다시 연결해 설명하세요.
2. 이 미니 실험은 `원 논문은 WMT'14 약 1,200만 문장 쌍, 160k/80k source/target vocabulary와 4층 1000-unit LSTM(약 3.8억 파라미터)을 여러 GPU에서 학습했다. 여기서는 8개 길이-3 수열을 한 층 LSTM으로 암기해 정보 흐름만 재현한다.`를 재현하지 않으므로 논문의 최종 성능을 검증한 것이 아닙니다.
3. 다음에는 데이터 크기, 모델 폭, 학습 횟수 중 **하나만** 바꾸고 같은 seed에서 metric을 비교하세요.

정답의 수치 자체보다, shape·확률 정규화·mask·gradient 같은 불변식을 먼저 확인하는 습관이
논문 구현에서 가장 강한 디버깅 도구입니다.